### Optional dependency setup
Run this cell only if imports fail in your environment.

In [ ]:
import importlib.util

packages = [
    'numpy', 'matplotlib', 'plotly', 'ipywidgets', 'scipy', 'pandas', 'sklearn', 'seaborn'
]
missing = [p for p in packages if importlib.util.find_spec(p) is None]
if missing:
    print('Installing missing packages:', missing)
    get_ipython().run_line_magic('pip', 'install -q ' + ' '.join(missing))
else:
    print('All common packages already available.')


# Interactive Hinge Loss Family
This notebook covers all three Lecture 3 hinge-loss figures: hinge loss, softplus approximation, and subgradient.

Use score form \((\theta^T x - b)\), where **b** is the bias term.


In [ ]:
# INTERACTIVE_WIDGET_SECTION
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

def hinge_loss(score, y=1.0, margin=0.0):
    return np.maximum(0.0, margin - y * score)

def softplus(score, y=1.0, margin=0.0, tau=0.3):
    z = (margin - y * score) / max(tau, 1e-8)
    return tau * np.log1p(np.exp(z))

def hinge_subgrad(score, y=1.0, margin=0.0):
    return np.where(margin - y * score > 0, -y, 0.0)

def plot_hinge_family(view='Hinge loss', b=0.0, margin=0.0, tau=0.3):
    theta_tx = np.linspace(-3.0, 3.0, 600)
    score = theta_tx - b  # score s = theta^T x - b

    plt.figure(figsize=(8, 5))
    colors = {1: 'tab:blue', -1: 'tab:red'}

    if view == 'Hinge loss':
        for y in (1, -1):
            h = hinge_loss(score, y=y, margin=margin)
            plt.plot(theta_tx, h, lw=2, color=colors[y], label=f'Hinge loss (y={y:+d})')
        plt.ylabel('loss')
        plt.title(r'Hinge Loss with Bias: $s=\theta^T x - b$')

    elif view == 'Hinge vs Softplus':
        for y in (1, -1):
            h = hinge_loss(score, y=y, margin=margin)
            sp = softplus(score, y=y, margin=margin, tau=tau)
            plt.plot(theta_tx, h, lw=2, color=colors[y], label=f'Hinge (y={y:+d})')
            plt.plot(theta_tx, sp, lw=2, color=colors[y], ls='--', label=f'Softplus (y={y:+d}, tau={tau:.2f})')
        plt.ylabel('loss')
        plt.title(r'Hinge vs Softplus with Bias: $s=\theta^T x - b$')

    else:
        for y in (1, -1):
            h = hinge_loss(score, y=y, margin=margin)
            sg = hinge_subgrad(score, y=y, margin=margin)
            plt.plot(theta_tx, h, lw=2, color=colors[y], ls='--', label=f'Hinge (y={y:+d})')
            plt.plot(theta_tx, sg, lw=2, color=colors[y], label=f'Subgradient (y={y:+d})')
        plt.ylabel('hinge / subgradient')
        plt.title(r'Hinge Subgradient with Bias: $s=\theta^T x - b$')

    plt.axvline(b, color='gray', ls=':', label=r'$\theta^T x=b$')
    for y in (1, -1):
        kink = b + margin / y
        plt.axvline(kink, color=colors[y], ls=':', alpha=0.8, label=f'margin kink (y={y:+d})')
    plt.xlabel(r'$\theta^T x$')
    plt.grid(True, alpha=0.3)
    plt.legend(ncol=2)
    plt.show()

widgets.interact(
    plot_hinge_family,
    view=widgets.ToggleButtons(
        options=['Hinge loss', 'Hinge vs Softplus', 'Subgradient'],
        value='Hinge loss',
        description='Figure'
    ),
    b=widgets.FloatSlider(value=0.0, min=-2.0, max=2.0, step=0.1, description='Bias b'),
    margin=widgets.FloatSlider(value=0.0, min=0.0, max=2.0, step=0.1, description='Margin'),
    tau=widgets.FloatSlider(value=0.3, min=0.05, max=1.5, step=0.05, description='Softplus tau')
)
